# Data Audit & Vector Analysis

This notebook verifies data quality and analyzes the extracted policy vector.

**Vector Method**: Positional windowing - extracts activations at every sentence boundary (1 to N sentences) from both on-policy and off-policy rollouts, then computes mean-difference vector.

**Key validations**:
1. Data statistics match expected counts
2. Vector properly captures on/off-policy distinction
3. No systematic length/truncation confounds
4. Layer-wise performance analysis
5. Sample distribution analysis

## 1. Setup

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('../data')
ARTIFACT_DIR = Path('../artifacts')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

## 2. Load and Validate Data

In [ ]:
# Load all data files
with open(DATA_DIR / 'on_policy.json') as f:
    on_policy_data = json.load(f)

with open(DATA_DIR / 'off_policy.json') as f:
    off_policy_data = json.load(f)

with open(DATA_DIR / 'sentence_paraphrases.json') as f:
    paraphrase_db = json.load(f)

# Load vector
with open(ARTIFACT_DIR / 'vector.json') as f:
    vector_data = json.load(f)

print("=" * 60)
print("DATA STATISTICS")
print("=" * 60)
print(f"Prompts: {len(on_policy_data)}")
print(f"On-policy rollouts: {sum(len(ex['rollouts']) for ex in on_policy_data)}")
print(f"Off-policy prompts: {len(off_policy_data)}")
print(f"Off-policy on-policy rollouts: {sum(len(ex['on_policy']) for ex in off_policy_data)}")
print(f"Off-policy variants: {sum(len(ex['off_policy']) for ex in off_policy_data)}")
print(f"Unique sentences paraphrased: {len(paraphrase_db)}")
print(f"Total paraphrases: {sum(len(v) for v in paraphrase_db.values())}")
print(f"\nSentences with all 3 paraphrases: {sum(1 for v in paraphrase_db.values() if len(v) == 3)} / {len(paraphrase_db)}")
print("\n" + "=" * 60)
print("VECTOR INFORMATION")
print("=" * 60)
print(f"Policy Vector:")
print(f"  Best Layer: {vector_data['layer']}")
print(f"  Method: Positional windowing (sentence boundaries)")
print(f"  Dimension: {len(vector_data['vector'])}")
print(f"  Layers considered: {vector_data['layers_considered'][0]}-{vector_data['layers_considered'][-1]}")
print(f"  Training samples: {vector_data['total_training_samples']}")